# 🧠 ReCurRAG — Recursive Language Model (RLM) Pipeline

This notebook implements and evaluates the **Recursive Language Model (RLM)** pipeline — an agentic, iterative reasoning system — across three distinct dataset types:

| # | Dataset | Type | Source |
|---|---------|------|--------|
| 1 | **arXiv Papers** | Long Documents (Unstructured) | Climate-Finance Research PDFs |
| 2 | **Wine Quality** | Semi-Structured (CSV/Tabular) | UCI ML Repository |
| 3 | **HotpotQA** | Multi-Hop QA (JSON) | EMNLP 2018 Benchmark |

### Key Difference from RAG
- **RAG**: `Query → Retrieve → Generate` (single-pass, linear)
- **RLM**: `Query → Plan → Tool Use → Reason → Refine → Aggregate` (iterative, multi-hop)

Results are stored in `outputs/rlm/` for comparison with the RAG pipeline.

## 📦 Setup & Imports

In [ ]:
import os
import sys
import json
import time
import pandas as pd
import numpy as np

# Suppress tokenizer parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Ensure project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

print(f"Project root: {project_root}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
from src.rlm.pipeline import RLMPipeline
from src.rlm.agent import RLMAgent
from src.rlm.tools import TOOL_DEFINITIONS, ToolExecutor

print("✅ All RLM modules imported successfully!")
print(f"   Available tools: {[t['function']['name'] for t in TOOL_DEFINITIONS]}")

## 📋 Configuration

In [ ]:
# Load query configuration
with open("configs/queries.json", "r") as f:
    queries_config = json.load(f)

# Dataset paths
DATASET_CONFIGS = {
    "long_docs": {
        "data_path": "data/raw/Long-Docs/papers/",
        "data_type": "long_docs",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "max_iterations": 8,
    },
    "semi_structured": {
        "data_path": "data/raw/Semi-Structured/wine+quality/",
        "data_type": "semi_structured",
        "chunk_size": 800,
        "chunk_overlap": 150,
        "max_iterations": 8,
    },
    "multi_hop": {
        "data_path": "data/raw/Multi-HopQA/hotpotqa.json",
        "data_type": "multi_hop",
        "chunk_size": 500,
        "chunk_overlap": 100,
        "max_iterations": 10,
    },
}

print("📋 Configuration loaded:")
for name, cfg in DATASET_CONFIGS.items():
    print(f"  {name}: {cfg['data_path']} (max_iter={cfg['max_iterations']})")

---
## 📄 Dataset 1: Long Documents (arXiv Papers)

The RLM agent uses **iterative tool calls** to explore the research papers more deeply than a single RAG retrieval pass. It can:
- First survey the available documents (`get_document_summary`)
- Search with multiple queries from different angles (`search_knowledge_base`)
- Synthesize intermediate findings (`reason_step`)
- Perform targeted follow-up searches to fill gaps

**Expected Advantage**: More comprehensive, well-cited answers with deeper context coverage.

In [ ]:
# Initialize and ingest long documents
rlm_long_docs = RLMPipeline(**DATASET_CONFIGS["long_docs"])
rlm_long_docs.ingest()

In [ ]:
# Display ingestion statistics
print("\n📊 Long-Docs Ingestion Summary (RLM):")
print(f"  Documents loaded: {rlm_long_docs.ingest_metadata['num_documents']}")
print(f"  Chunks created:   {rlm_long_docs.ingest_metadata['num_chunks']}")
print(f"  Max iterations:   {rlm_long_docs.ingest_metadata['max_iterations']}")
print(f"  Model:            {rlm_long_docs.ingest_metadata['model']}")

In [ ]:
# Run queries on Long Documents
long_docs_questions = queries_config["long_docs"]["questions"]

print(f"🔍 Running {len(long_docs_questions)} queries on Long-Docs (RLM)...\n")
long_docs_results = rlm_long_docs.run_batch(long_docs_questions)

# Display results with reasoning traces
for i, result in enumerate(long_docs_results):
    print(f"\n{'─'*60}")
    print(f"Q{i+1}: {result['question']}")
    print(f"A:  {result['answer'][:300]}..." if len(result['answer']) > 300 else f"A:  {result['answer']}")
    print(f"⏱️  Latency: {result['latency_s']}s | Iterations: {result['num_iterations']} | "
          f"Tool Calls: {result['total_tool_calls']} | Depth: {result['reasoning_depth']}")

In [ ]:
# Save Long-Docs results
long_docs_output_path = rlm_long_docs.save_results(long_docs_results)
print(f"\n✅ Long-Docs RLM results saved to: {long_docs_output_path}")

---
## 📊 Dataset 2: Semi-Structured Data (Wine Quality)

For structured data, the RLM agent has access to the `analyze_data` tool which performs **real statistical analysis** on the raw CSV data — computing correlations, distributions, and comparisons that simple text retrieval cannot provide.

**Expected Advantage**: Precise, data-driven answers with specific numbers instead of RAG's text-matching approach.

In [ ]:
# Initialize and ingest semi-structured data
rlm_semi = RLMPipeline(**DATASET_CONFIGS["semi_structured"])
rlm_semi.ingest()

In [ ]:
# Run queries on Semi-Structured data
semi_questions = queries_config["semi_structured"]["questions"]

print(f"🔍 Running {len(semi_questions)} queries on Semi-Structured data (RLM)...\n")
semi_results = rlm_semi.run_batch(semi_questions)

# Display results
for i, result in enumerate(semi_results):
    print(f"\n{'─'*60}")
    print(f"Q{i+1}: {result['question']}")
    print(f"A:  {result['answer'][:300]}..." if len(result['answer']) > 300 else f"A:  {result['answer']}")
    print(f"⏱️  Latency: {result['latency_s']}s | Tools: {result['tool_call_breakdown']}")

In [ ]:
# Save Semi-Structured results
semi_output_path = rlm_semi.save_results(semi_results)
print(f"\n✅ Semi-Structured RLM results saved to: {semi_output_path}")

---
## 🔗 Dataset 3: Multi-Hop QA (HotpotQA)

Multi-hop questions are where the RLM truly shines. These questions **cannot** be answered with a single retrieval pass — they require:
1. Breaking the question into sub-questions
2. Searching for each entity separately
3. Connecting evidence across documents
4. Drawing intermediate conclusions

**Expected Advantage**: Significantly higher Exact Match and F1 scores compared to RAG.

In [ ]:
# Initialize and ingest Multi-Hop QA data
rlm_multi_hop = RLMPipeline(**DATASET_CONFIGS["multi_hop"])
rlm_multi_hop.ingest()

In [ ]:
# Run HotpotQA evaluation
MAX_EVAL_SAMPLES = 50

print(f"🔍 Running HotpotQA evaluation ({MAX_EVAL_SAMPLES} samples, RLM)...\n")
multi_hop_results = rlm_multi_hop.run_hotpotqa_evaluation(max_samples=MAX_EVAL_SAMPLES)

In [ ]:
# Display Multi-Hop results with ground truth comparison
print("\n📊 Multi-Hop QA Results (RLM — first 10):")
print(f"{'─'*80}")

for i, result in enumerate(multi_hop_results[:10]):
    print(f"\nQ{i+1}: {result['question']}")
    print(f"  RLM Answer:    {result['answer'][:200]}")
    print(f"  Ground Truth:  {result['ground_truth_answer']}")
    print(f"  Level: {result['level']} | Iterations: {result['num_iterations']} | "
          f"Tools: {result['total_tool_calls']} | Depth: {result['reasoning_depth']}")
    print(f"{'─'*80}")

In [ ]:
# Save Multi-Hop results
multi_hop_output_path = rlm_multi_hop.save_results(multi_hop_results)
print(f"\n✅ Multi-Hop QA RLM results saved to: {multi_hop_output_path}")

---
## 📈 RLM Performance Summary

In [ ]:
# Compute summary statistics
summary_data = []

datasets = {
    "Long Documents": long_docs_results,
    "Semi-Structured": semi_results,
    "Multi-Hop QA": multi_hop_results,
}

for name, results in datasets.items():
    latencies = [r["latency_s"] for r in results]
    tool_calls = [r["total_tool_calls"] for r in results]
    depths = [r["reasoning_depth"] for r in results]
    iterations = [r["num_iterations"] for r in results]
    
    row = {
        "Dataset": name,
        "Queries": len(results),
        "Avg Latency (s)": round(np.mean(latencies), 3),
        "Avg Tool Calls": round(np.mean(tool_calls), 1),
        "Avg Reasoning Depth": round(np.mean(depths), 1),
        "Avg Iterations": round(np.mean(iterations), 1),
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print("\n📊 RLM Pipeline — Performance Summary")
print("="*70)
display(summary_df)

In [ ]:
# Save combined RLM summary
os.makedirs("outputs/rlm", exist_ok=True)

combined_summary = {
    "pipeline": "rlm",
    "datasets": {},
}

for ds_name, result_file in [
    ("long_docs", "outputs/rlm/long_docs/long_docs_results.json"),
    ("semi_structured", "outputs/rlm/semi_structured/semi_structured_results.json"),
    ("multi_hop", "outputs/rlm/multi_hop/multi_hop_results.json"),
]:
    if os.path.exists(result_file):
        with open(result_file, "r") as f:
            data = json.load(f)
        combined_summary["datasets"][ds_name] = data["summary"]

with open("outputs/rlm/rlm_summary.json", "w") as f:
    json.dump(combined_summary, f, indent=2)

print("💾 Combined RLM summary saved to: outputs/rlm/rlm_summary.json")
print(json.dumps(combined_summary, indent=2))

---
## 🔍 Output Structure

```
outputs/
├── rag/                                # RAG results (from rag_pipeline.ipynb)
│   ├── long_docs/
│   ├── semi_structured/
│   └── multi_hop/
├── rlm/                                # RLM results (from this notebook)
│   ├── rlm_summary.json
│   ├── long_docs/
│   │   └── long_docs_results.json
│   ├── semi_structured/
│   │   └── semi_structured_results.json
│   └── multi_hop/
│       └── multi_hop_results.json
└── comparison_report.json              # Generated by main.py
```

### Next Steps
Run the comparison dashboard:
```bash
python main.py              # Runs evaluation + launches dashboard
python main.py --eval-only  # Just generates comparison_report.json
```

In [ ]:
# Verify output files exist
print("📁 Output files generated:")
for root, dirs, files in os.walk("outputs/rlm"):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        print(f"  {filepath} ({size:,} bytes)")

print("\n✅ RLM pipeline complete! Ready for evaluation.")